## Classified Project: "BIRD IS THE WORD"
**(or just birdstrikes - a look at the data - course: Data Analysis & Visualizaion)**

**Initial thoughts (some lose asumptions)**

In answering the first question of when during the 24h cycle and what phase of flight strikes occur. I'm taking the following into account:

'Phase of flight'

The 'Time of day' column contains 4 unique values.

Time of year can be derived from the "Flight Date" column. Here I would like to examine if bird migration has any impact on birdstrikes. Which brings us to the next column in the DataFrame.**

Locations can be derived from the'Origin State' column.

In [ ]:
# Importing the relevand libraries.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Applying Seaborn's darkgrid theme.
sns.set_theme(style='darkgrid')

# Creating a python list containing column names with dates for parsing.
strikes_raw = pd.read_csv(
    'data/birdstrikes.csv',
    parse_dates=['Flight Date']
)

# Loading the CSV and parsing the date_columns for the proper datatype in the DataFrame.
regions_raw = pd.read_csv(
    'data/us_regions.csv'
)

strikes = strikes_raw.copy()
regions = regions_raw.copy()



In [ ]:
# Displaying the number of rows and columns in the DataFrame.
print('Bird strikes:', strikes_raw.shape)

In [ ]:
strikes_raw.dtypes

In [ ]:
strikes_raw.head()

In [ ]:
strikes_raw.info()

In [ ]:
# Inspecting the parsed column separately.
print(strikes['Flight Date'].dtype)

In [ ]:
before_parse_date = strikes_raw['Flight Date'].isna().sum()
missing_after = strikes['Flight Date'].isna().sum()

# There are apparently no NaT values (missing dates) during conversion (after parsing the Flight Date column).
print('Missing before:', before_parse_date)
print('Missing after:', missing_after)
print('New missing dates:', missing_after - before_parse_date)

In [ ]:
print('Earliest date:', strikes['Flight Date'].min())
print('Latest date:', strikes['Flight Date'].max())
# strikes['Flight Date'].describe()

In [ ]:
# Checking for duplicate rows
duplicate_after_first = strikes.duplicated().sum()

duplicate_rows = strikes[strikes.duplicated(keep=False)]

print('Duplicate rows after the first occurrence:', duplicate_after_first)
print('All rows involved in duplicate groups:', len(duplicate_rows))
print('Distinct records that are repeated:', len(duplicate_rows.drop_duplicates()))

**Anwering the first ("1.") requirement in the project PDF ("Investigate data"):**

The table consists of 10K rows and 14 columns.

Unit of observation (or grain of the data): In this table, one row is one reported collision between an aircraft and a bird (or "wildlife" since they are also not all birds).

I will mainly use `Flight Date`, `Phase of flight` and `Time of day` to examine 
when reported strikes occurred.

I will use `Effect Amount of damage`, `Cost Other`, `Cost Repair` and `Cost Total $` to investigate recorded damage and financial cost.

`Wildlife Size` and `Wildlife Species` will help compare 
the animals involved, while `Speed IAS in knots` may help describe the flight 
conditions.

I will use `Origin State` to merge the main table with 
`us_regions.csv`, creating a `Region` column for regional comparisons. 

`Airport Name` will provide location context and may be used for selected 
airport-level summaries.

In [ ]:
# Cleaning duplicate rows
strikes = (
    strikes
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_before_duplicates = len(strikes_raw) # Checking number of rows by querying the untouched dataframe.
print('Number of rows before duplicate rows removal:', rows_before_duplicates)

print('Number of rows after duplicate rows removal:', len(strikes))

**On the step of removing duplicate rows**
I found 24 exact duplicate rows after the first occurrence. Because all 14 fields matched, I treated the duplicates as redundant records and removed them, reducing the table from 10,000 to 9,976 rows. The table has no unique incident ID, so I cannot prove that the matching rows represent the same collision. They could represent separate incidents with identical data.

___

Moving on to the next steps of cleaning our strikes dataframe following duplicate rows removal.

In [ ]:
# Checking missing values for each column (after duplicate rows removal)
missing_summary = pd.DataFrame({
    'missing_count': strikes.isna().sum(),
    'missing_percent': (
        strikes.isna().mean() * 100
    ).round(1)
}).sort_values(
    'missing_percent',
    ascending=False
)

missing_summary

In [ ]:
# Missing does not prove that no damage occurred. Avoid treating this field as a complete measurement of damage severity.
strikes['Effect Amount of damage'].value_counts(dropna=False)

In [ ]:
# Making a comparison between missing damage descriptions and cost.
damage_missing = (strikes['Effect Amount of damage'].isna())

damage_cost_check = (
    strikes
    .groupby(damage_missing)['Cost Total $']
    .agg(['count', 'median', 'mean', 'max'])
    .round(2)
)

damage_cost_check.index = ['Damage description recorded', 'Damage description missing']

damage_cost_check

In [ ]:
# Missing description does not necessarily mean zero economic effect.
missing_damage_positive_cost = (
    damage_missing
    & (strikes['Cost Total $'] > 0)
)

print(
    'Missing damage description but positive total cost:',
    missing_damage_positive_cost.sum()
)

In [ ]:
# Adding a column to the working DataFrame (strikes).
strikes['Damage Description Recorded'] = (
    strikes['Effect Amount of damage'].notna()
)
print("Before adding 'Damage Description Recorded':", strikes_raw.shape)
print('After adding the column:', strikes.shape)

**A comment on the cleaning step above**
The "damage-description" field is missing for a majority of the strike reports. Rather than "inventing information", I left these values missing (for instance, I did not populate these values with “No damage”). Some reports with a missing damage description still have a positive recorded cost, showing that absence of a category or description is not equivalent to absence of damage.

___

**Next step in cleaning the data:** Categories C (14 of them) and B (1 entry) do not match the readable categories Minor, Medium, and Substantial. This is cause for further examination of the data.

In [ ]:
unclear_damage_codes = strikes[strikes['Effect Amount of damage'].isin(['B', 'C'])]

display(
    unclear_damage_codes[
        [
            'Flight Date',
            'Airport Name',
            'Effect Amount of damage',
            'Cost Total $'
        ]
    ]
)

print('Rows with unexplained B or C codes:', len(unclear_damage_codes))

In [ ]:
# Adding column 'Damage Category' for analysis. "Grouping" B and C under "Unclear damage code". Prevents presenting unexplained codes as if their meaning were known (when they are not known to us at present).
strikes['Damage Category'] = (
    strikes['Effect Amount of damage']
    .replace({
        'B': 'Unclear code',
        'C': 'Unclear code'
    })
    .fillna('Not recorded')
)

strikes['Damage Category'].value_counts()

Fifteen reports use the unexplained values B or C instead of descriptive readable damage categories. I preserved the original field as 'Damage Category' and created a separate analysis category called Unclear code. I did not translate the letters because I could not verify their meaning for this dataset version.
___

In [ ]:
speed_missing_count = (
    strikes['Speed IAS in knots'].isna().sum()
)

speed_missing_percent = (
    strikes['Speed IAS in knots']
    .isna()
    .mean()
    * 100
)

print('Missing speeds:', speed_missing_count)
print(
    'Missing percentage:',
    round(speed_missing_percent, 1)
)

In [ ]:
speed_missing_by_phase = pd.crosstab(
    strikes['Phase of flight'],
    strikes['Speed IAS in knots'].isna(),
    normalize='index'
) * 100

speed_missing_by_phase.columns = [
    'Speed recorded %',
    'Speed missing %'
]

speed_missing_by_phase.round(1)

In [ ]:
# An indicator (column) telling us if any speed entries are available (over rows).
strikes['Speed Recorded'] = (
    strikes['Speed IAS in knots'].notna()
)

# # Filtering recorded values for analysis.
# speed_known = strikes[
#     strikes['Speed IAS in knots'].notna()
# ].copy()

Airspeed is missing in a substantial share of reports. I retained the missing values because zero would mean a stationary aircraft, while mean or median filling would create fictional speeds and reduce the observed spread. Any analyses involving speed therefore use only reports with a recorded value and state the number of observations used.


In [ ]:
zero_speed = (
    strikes['Speed IAS in knots'] == 0
)

print('Rows with zero recorded speed, all', zero_speed.sum(), "of them. :)")

display(
    strikes.loc[
        zero_speed,
        [
            'Flight Date',
            'Airport Name',
            'Phase of flight',
            'Speed IAS in knots'
        ]
    ]
)

In [ ]:
# Creating a column, indicating questionable combinations between speed and "phase of flight".
strikes['Questionable Zero Speed'] = (
    (strikes['Speed IAS in knots'] == 0)
    & ~strikes['Phase of flight'].isin(
        ['Parked', 'Taxi', 'Landing Roll']
    )
)

In [ ]:
# Actually showing what we are talking about in regard to questionable combinations of speed and phase. It can also be mentioned that this is not an official aviation rule but merely for analysis.
print(
    'Questionable zero-speed rows:',
    strikes['Questionable Zero Speed'].sum()
)

display(
    strikes.loc[
        strikes['Questionable Zero Speed'],
        [
            'Flight Date',
            'Airport Name',
            'Phase of flight',
            'Speed IAS in knots'
        ]
    ]
)

In [ ]:
# Separate observation: Index 19 (row 20) shows NaN and not zero (or 0).
strikes['Speed IAS in knots'].notna().head(20)

In [ ]:
# Costs show consistency across the rows.
cost_mismatch = (
    strikes['Cost Total $']
    != (
        strikes['Cost Other']
        + strikes['Cost Repair']
    )
)

print(
    'Rows where total differs from its components:',
    cost_mismatch.sum()
)

In [ ]:
zero_cost = strikes['Cost Total $'] == 0
positive_cost = strikes['Cost Total $'] > 0

print('Zero recorded total cost:', zero_cost.sum())
print('Positive recorded total cost:', positive_cost.sum())

print(
    'Zero cost percentage:',
    round(zero_cost.mean() * 100, 1)
)

The dataset page identifies all three cost columns as numeric fields containing zero values, but provides no field-level explanation of whether zero means confirmed no cost or no cost supplied. Another analysis of FAA data explicitly treats missing cost as zero in sums and says it does not impute missing cost, illustrating why cost interpretation needs methodological caution.

In [ ]:
# Compairing 'Damage Category' with cost.
cost_damage_table = pd.crosstab(
    strikes['Damage Category'],
    zero_cost,
    margins=True
)

cost_damage_table.columns = [
    'Positive cost',
    'Zero cost',
    'Total'
]

cost_damage_table

It can be noted that some reports marked Minor, Medium, or Substantial still have a zero recorded cost. Therefore, zero cannot safely be interpreted as "nothing happened". I'll therefore leave the original cost values unchanged.

In [ ]:
strikes['Positive Recorded Cost'] = (
    strikes['Cost Total $'] > 0
)

In [ ]:
print(
    'Median across all reports:',
    strikes['Cost Total $'].median()
)

print(
    'Median among positive recorded costs:',
    strikes.loc[
        strikes['Cost Total $'] > 0,
        'Cost Total $'
    ].median()
)

Most reports contain a total cost of zero, but zero may represent either a genuine zero cost or the absence of a supplied cost estimate. I retained the original values and created a flag for positive recorded cost. I will therefore describe zero as “zero recorded cost,” not as proof that the incident caused no financial loss.

In [ ]:
category_columns = [
    'Phase of flight',
    'Time of day',
    'Wildlife Size',
    'Origin State',
    'Effect Amount of damage'
]

for column in category_columns:
    print(f'\n--- {column} ---')
    print(
        strikes[column]
        .value_counts(dropna=False)
    )

In [ ]:
# Checking if there is any hidden whitespace to be cleaned out (seemingly none).
for column in category_columns:
    non_missing = strikes[column].dropna()

    whitespace_count = (
        non_missing
        != non_missing.str.strip()
    ).sum()

    print(
        column,
        '- values with surrounding whitespace:',
        whitespace_count
    )

I inspected the main categorical columns for inconsistent capitalization, spelling and surrounding whitespace. The phase-of-flight, time-of-day, wildlife-size and state labels were internally consistent. The damage field was the exception because it contained missing values and 15 unexplained (B and C) letter codes.
No values need correction in four of the columns.

In [ ]:
print('Original shape:', strikes_raw.shape)
print('Cleaned shape:', strikes.shape)

print(
    'Remaining exact duplicates:',
    strikes.duplicated().sum()
)

print('\nData types:')
print(strikes.dtypes)

print('\nMissing values:')
print(
    strikes.isna()
    .sum()
    .sort_values(ascending=False)
)

print('\nCreated columns:')
created_columns = [
    'Damage Description Recorded',
    'Damage Category',
    'Speed Recorded',
    'Questionable Zero Speed',
    'Positive Recorded Cost'
]

print(created_columns)

In [ ]:
# Varifying that the original columns are still present in the working DataFrame.
print(
    'Original damage field still present:',
    'Effect Amount of damage' in strikes.columns
)

print(
    'Original cost field still present:',
    'Cost Total $' in strikes.columns
)

One final reservation remains: because the dataset has no unique incident ID, even completely identical rows cannot be proven to be the same real-world collision. Removing them is defensible, but it should be described as a documented assumption rather than absolute certainty.

## For now, that's it for cleaning this table of birdstrikes. Moving on to "next steps"...

___
___

In [ ]:
# strikes['Year'] = strikes['Flight Date'].dt.year # Extracts year, resulting in the added 'Year' column of datatype int32.

# # Adding a column of dtype bool indicating where cost is registered.
# strikes['Positive Recorded Cost'] = (
#     strikes['Cost Total $'] > 0
# )

# strikes.head()

___
**Checking requirements for step 4 in the project PDF below.**

In [ ]:
# Inspecting the second table with the required merge in mind.
print(regions.shape)
display(regions.head())
regions.info()

The second table assigns each recorded origin state to one of four US
regions. One row in this table should represent one unique state-to-region
mapping. I will merge it with the strike table using `Origin State`.

In [ ]:
# Inspecting missing keys in both tables.
print(
    'Missing Origin State values in (bird)strikes:',
    strikes['Origin State'].isna().sum()
)

print(
    'Missing Origin State values in regions:',
    regions['Origin State'].isna().sum()
)

In [ ]:
# Checking whether Origin State is unique in the region table to mitigate unwanted extra rows at merge.
print(
    'Duplicate states in the region table:',
    regions['Origin State'].duplicated().sum()
)

print(
    'Is Origin State unique in the region table (?):',
    regions['Origin State'].is_unique
)

In [ ]:
# Checking which states appear in the strike table but have no matching state in the region table? The merge requires exact text matches, so, compairing the state names in both tables:

strike_states = set(
    strikes['Origin State'].dropna().unique()
)

region_states = set(
    regions['Origin State'].dropna().unique()
)

states_missing_from_lookup = (
    strike_states - region_states
)

unused_lookup_states = (
    region_states - strike_states
)

print(
    'Strike states missing from region table:',
    states_missing_from_lookup
)

print(
    'Region-table states unused by strikes:',
    unused_lookup_states
)

In [ ]:
# Compairing normalized text for both tables. Can detect same or similar meaning with different descriptive text, like 'DC' vs 'District of Columbia'.
print(
    strikes['Origin State']
    .dropna()
    .sort_values()
    .unique()
)

print(
    regions['Origin State']
    .dropna()
    .sort_values()
    .unique()
)

In [ ]:
rows_before_merge = len(strikes)

cost_before_merge = strikes['Cost Total $'].sum()

# Record one or more totals that should not change when a region label is added.
positive_cost_reports_before = (
    strikes['Cost Total $'] > 0
).sum()

print('Rows before merge:', rows_before_merge)
print('Total recorded cost before:', cost_before_merge)
print(
    'Positive-cost reports before:',
    positive_cost_reports_before
)

In [ ]:
# Performing the merge (using validate).
strikes_merged = strikes.merge(
    regions,
    on='Origin State',
    how='left',
    validate='many_to_one',
    indicator=True
)

In [ ]:
# Checking the resulting merge.
rows_after_merge = len(strikes_merged)

print('Shape before:', strikes.shape)
print('Shape after:', strikes_merged.shape)

print('Rows before:', rows_before_merge)
print('Rows after:', rows_after_merge)
print(
    'Row-count difference:',
    rows_after_merge - rows_before_merge
)

In [ ]:
print(
    strikes_merged['_merge']
    .value_counts(dropna=False)
)

In [ ]:
# Inspecting potential mismatches
unmatched_strikes = strikes_merged[
    strikes_merged['_merge'] == 'left_only'
]

display(
    unmatched_strikes[
        [
            'Origin State',
            'Airport Name',
            'Flight Date'
        ]
    ]
)

print(
    'Unmatched strike rows:',
    len(unmatched_strikes)
)

print(
    'Unmatched state labels:',
    unmatched_strikes['Origin State']
    .dropna()
    .unique()
)

In [ ]:
# Check control totals post merge.

cost_after_merge = (
    strikes_merged['Cost Total $'].sum()
)

positive_cost_reports_after = (
    strikes_merged['Cost Total $'] > 0
).sum()

print('Total recorded cost before:', cost_before_merge)
print('Total recorded cost after:', cost_after_merge)

print(
    'Positive-cost reports before:',
    positive_cost_reports_before
)

print(
    'Positive-cost reports after:',
    positive_cost_reports_after
)

In [ ]:
print(
    'Row count unchanged:',
    rows_before_merge == rows_after_merge
)

print(
    'Cost total unchanged:',
    cost_before_merge == cost_after_merge
)

print(
    'Positive-cost count unchanged:',
    positive_cost_reports_before
    == positive_cost_reports_after
)

Step 8

In [ ]:
print(
    strikes_merged['Region']
    .value_counts(dropna=False)
)
print(
    'Missing regions:',
    strikes_merged['Region'].isna().sum()
)

In [ ]:
strikes_merged = strikes_merged.drop(columns='_merge')

strikes = strikes_merged.copy()

# Simpler variant of the same drop
# strikes = strikes_merged.drop(
#     columns='_merge'
# )


In [ ]:
print(strikes.shape)
print(strikes.columns)
display(
    strikes[
        [
            'Origin State',
            'Region'
        ]
    ].head(20)
)

In [ ]:
strikes_merged.columns

___

**Comment post merge with the US region table**

The strike table contains many reports from the same state, while the region
table contains one region assignment per state. I therefore used a
many-to-one left merge on `Origin State`. A left merge preserved every strike
report, while `validate='many_to_one'` checked that each state appeared no
more than once in the lookup table.

The number of strike rows was unchanged after the merge, all rows matched a
region, and the total recorded cost and number of positive-cost reports were
also unchanged. The merge therefore added regional information without
removing or multiplying strike records.

In [ ]:
# Extract the calendar year for yearly comparisons.
strikes['Year'] = strikes['Flight Date'].dt.year

In [ ]:
# Create a cautious category describing recorded costs.
strikes['Recorded Cost Status'] = (
    strikes['Cost Total $']
    .gt(0)
    .map({
        True: 'Positive recorded cost',
        False: 'Zero recorded cost'
    })
)

In [ ]:
# Create an analysis-friendly damage category while preserving the original damage field.
strikes['Damage Category'] = (
    strikes['Effect Amount of damage']
    .replace({
        'B': 'Unclear code',
        'C': 'Unclear code'
    })
    .fillna('Not recorded')
)

In [ ]:
# Check the new columns.
display(
    strikes[
        [
            'Flight Date',
            'Year',
            'Cost Total $',
            'Recorded Cost Status',
            'Effect Amount of damage',
            'Damage Category'
        ]
    ].head(10)
)

In [ ]:
print('\nReports per year:')
print(
    strikes['Year']
    .value_counts()
    .sort_index()
)

print('\nRecorded cost status:')
print(
    strikes['Recorded Cost Status']
    .value_counts(dropna=False)
)

print('\nDamage categories:')
print(
    strikes['Damage Category']
    .value_counts(dropna=False)
)

**New analytical columns**

I created `Year` from `Flight Date` so that I can compare the number of
reported wildlife strikes over time. This column represents the year of the
reported incident.

I created `Recorded Cost Status` to separate reports with a positive recorded
total cost from reports with a recorded total cost of zero. I use the wording
“zero recorded cost” rather than “no cost” because the table does not show
whether every zero is a confirmed absence of cost or an amount that was not
reported.

I also created `Damage Category` from `Effect Amount of damage`. Missing
descriptions were labelled `Not recorded`, while the unexplained values `B`
and `C` were grouped as `Unclear code`. The original damage column was
preserved so that no source information was overwritten.

___

# Requirement 6: Group and compare

In [ ]:
# Summary 1: reports and recorded speeds by phase of flight.
phase_summary = (
    strikes
    .groupby('Phase of flight')
    .agg(
        reported_strikes=('Flight Date', 'size'),
        median_speed=('Speed IAS in knots', 'median'),
        recorded_speeds=('Speed IAS in knots', 'count')
    )
    .sort_values(
        'reported_strikes',
        ascending=False
    )
    .round(1)
)

print('Reports by phase of flight:')
display(phase_summary)


In [ ]:
strikes['Positive Recorded Cost'] = (
    strikes['Cost Total $'] > 0
)

# Summary 2: reports and recorded costs by region.
region_summary = (
    strikes
    .groupby('Region')
    .agg(
        reported_strikes=('Flight Date', 'size'),
        positive_cost_reports=(
            'Positive Recorded Cost',
            'sum'
        ),
        total_recorded_cost=(
            'Cost Total $',
            'sum'
        ),
        median_recorded_cost=(
            'Cost Total $',
            'median'
        )
    )
)

region_summary[
    'positive_cost_percent'
] = (
    region_summary['positive_cost_reports']
    / region_summary['reported_strikes']
    * 100
)

region_summary = (
    region_summary
    .sort_values(
        'reported_strikes',
        ascending=False
    )
    .round(1)
)

print('Regional comparison:')
display(region_summary)

In [ ]:
strikes['Damage Description Recorded'] = (
    strikes['Effect Amount of damage'].notna()
)

# Summary 3: reports, recorded costs and damage information
# by wildlife size.
wildlife_size_summary = (
    strikes
    .groupby('Wildlife Size')
    .agg(
        reported_strikes=('Flight Date', 'size'),
        positive_cost_reports=(
            'Positive Recorded Cost',
            'sum'
        ),
        median_recorded_cost=(
            'Cost Total $',
            'median'
        ),
        mean_recorded_cost=(
            'Cost Total $',
            'mean'
        ),
        maximum_recorded_cost=(
            'Cost Total $',
            'max'
        ),
        damage_descriptions_recorded=(
            'Damage Description Recorded',
            'sum'
        )
    )
)

wildlife_size_summary[
    'positive_cost_percent'
] = (
    wildlife_size_summary[
        'positive_cost_reports'
    ]
    / wildlife_size_summary[
        'reported_strikes'
    ]
    * 100
)

wildlife_size_summary[
    'damage_description_percent'
] = (
    wildlife_size_summary[
        'damage_descriptions_recorded'
    ]
    / wildlife_size_summary[
        'reported_strikes'
    ]
    * 100
)

wildlife_size_summary = (
    wildlife_size_summary
    .sort_values(
        'reported_strikes',
        ascending=False
    )
    .round(1)
)

print('Comparison by wildlife size:')
display(wildlife_size_summary)

**Comparison by wildlife size**

I grouped the reports by wildlife size and compared report counts, positive
recorded costs and the availability of a damage description. I included both
the mean and median recorded cost because a small number of very expensive
reports may pull the mean upward.

A zero median does not prove that a typical incident caused no cost. It means
that at least half the reports in that group have a recorded cost of zero,
which may represent either a genuine zero or an unreported amount.

___

## Requirement 7: Describe recorded costs in depth

In [ ]:
cost = strikes['Cost Total $']

# General descriptive statistics.
print('All recorded costs:')
print(
    cost.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).round(2)
)

print('\nMean:', cost.mean().round(1))
print('Median:', cost.median())
print('Standard deviation:', cost.std().round(1))

# Zero versus positive recorded costs.
zero_cost_count = (cost == 0).sum()
positive_cost_count = (cost > 0).sum()

print('\nZero recorded costs:', zero_cost_count)
print(
    'Zero recorded cost percentage:',
    ((cost == 0).mean() * 100).round(1)
)

print('Positive recorded costs:', positive_cost_count)
print(
    'Positive recorded cost percentage:',
    ((cost > 0).mean() * 100).round(1)
)


In [ ]:
# IQR rule for all reports.
q1 = cost.quantile(0.25)
q3 = cost.quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

cost_outliers = (
    (cost < lower_fence)
    | (cost > upper_fence)
)

print('\nAll-report IQR:')
print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower fence:', lower_fence)
print('Upper fence:', upper_fence)
print('Outliers:', cost_outliers.sum())

In [ ]:
# Analyse positive recorded costs separately.
positive_costs = strikes.loc[
    strikes['Cost Total $'] > 0,
    'Cost Total $'
]

print('\nPositive recorded costs:')
print(
    positive_costs.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).round(2)
)

positive_q1 = positive_costs.quantile(0.25)
positive_q3 = positive_costs.quantile(0.75)
positive_iqr = positive_q3 - positive_q1

positive_lower_fence = (
    positive_q1 - 1.5 * positive_iqr
)

positive_upper_fence = (
    positive_q3 + 1.5 * positive_iqr
)

positive_cost_outliers = (
    (positive_costs < positive_lower_fence)
    | (positive_costs > positive_upper_fence)
)

print('\nPositive-cost IQR:')
print('Q1:', positive_q1)
print('Q3:', positive_q3)
print('IQR:', positive_iqr)
print('Lower fence:', positive_lower_fence)
print('Upper fence:', positive_upper_fence)
print(
    'Outliers:',
    positive_cost_outliers.sum()
)


In [ ]:
# Compare costs by wildlife size.
cost_by_size = (
    strikes
    .groupby('Wildlife Size')
    .agg(
        reported_strikes=(
            'Cost Total $',
            'size'
        ),
        positive_cost_reports=(
            'Positive Recorded Cost',
            'sum'
        ),
        mean_recorded_cost=(
            'Cost Total $',
            'mean'
        ),
        median_recorded_cost=(
            'Cost Total $',
            'median'
        ),
        maximum_recorded_cost=(
            'Cost Total $',
            'max'
        )
    )
)

cost_by_size[
    'positive_cost_percent'
] = (
    cost_by_size['positive_cost_reports']
    / cost_by_size['reported_strikes']
    * 100
)

cost_by_size = (
    cost_by_size
    .sort_values(
        'positive_cost_percent',
        ascending=False
    )
    .round(1)
)

print('\nRecorded costs by wildlife size:')
display(cost_by_size)


In [ ]:
# Inspect the largest individual recorded costs.
largest_costs = (
    strikes.loc[
        strikes['Cost Total $'] > 0,
        [
            'Flight Date',
            'Airport Name',
            'Wildlife Species',
            'Wildlife Size',
            'Damage Category',
            'Cost Total $'
        ]
    ]
    .sort_values(
        'Cost Total $',
        ascending=False
    )
    .head(10)
)

print('\nLargest recorded costs:')
display(largest_costs)


**Distribution of recorded costs**

Most reports have a recorded total cost of zero: 9767 reports, or
97,9% percent of the cleaned table. Consequently, Q1, the median and
Q3 are all zero. The IQR is therefore zero, and the standard IQR rule classifies
every positive recorded cost as an outlier. In this case, the IQR rule mainly
separates zero from positive values and does not distinguish moderately
expensive reports from exceptionally expensive ones.

Among reports with a positive recorded cost, the median is 23325 USD, while
the mean is 193996.54 USD. The mean is higher because a small number of extremely
expensive reports create a long right tail. I therefore consider the median
the better description of a typical positive recorded cost, while the mean
is more useful for describing the total financial burden.

The share of reports with a positive recorded cost differs by wildlife size.
However, the values represent reported costs only. A recorded zero may mean
either a genuine zero or the absence of a supplied amount, so the results
should not be interpreted as the complete economic cost of wildlife strikes.

___

**Requirement 8: Draw the data**

In [ ]:
sns.set_theme(style='darkgrid')

# CHART 1: Horizontal bar chart by phase of flight.
phase_counts = (
    strikes['Phase of flight']
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    phase_counts.index,
    phase_counts.values
)

ax.set_title(
    'Most Reported Wildlife Strikes Occurred During Approach'
)
ax.set_xlabel('Number of reported strikes')
ax.set_ylabel('Phase of flight')

plt.tight_layout()
plt.show()


In [ ]:
# CHART 2: Heatmap of phase and time of day.
phase_time_table = pd.crosstab(
    strikes['Phase of flight'],
    strikes['Time of day']
)

fig, ax = plt.subplots(figsize=(9, 6))

sns.heatmap(
    phase_time_table,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=ax
)

ax.set_title(
    'Reported Wildlife Strikes by Flight Phase and Time of Day'
)
ax.set_xlabel('Time of day')
ax.set_ylabel('Phase of flight')

plt.tight_layout()
plt.show()

In [ ]:
# CHART 3: Line chart by year.
reports_by_year = (
    strikes
    .groupby('Year')
    .size()
)

fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    x=reports_by_year.index,
    y=reports_by_year.values,
    marker='o',
    ax=ax
)

ax.set_title(
    'Number of Reported Wildlife Strikes Changed Over Time'
)
ax.set_xlabel('Year')
ax.set_ylabel('Number of reported strikes')

plt.tight_layout()
plt.show()

In [ ]:
# CHART 4: Histogram of positive recorded costs.
cost_display_limit = (
    positive_costs.quantile(0.95)
)

costs_for_display = positive_costs[
    positive_costs <= cost_display_limit
]

excluded_costs = (
    positive_costs > cost_display_limit
).sum()

fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    costs_for_display,
    bins=30,
    ax=ax
)

ax.set_title(
    f'Distribution of Positive Recorded Costs '
    f'({excluded_costs} Above the Displayed Range)'
)
ax.set_xlabel('Positive recorded total cost in US dollars')
ax.set_ylabel('Number of reports')

plt.tight_layout()
plt.show()


CHART 4: Most positive recorded costs fall toward the lower end of the displayed range, while eleven (11) larger values are omitted from the chart so that the main distribution remains visible.

In [ ]:
# CHART 4: Logarithmic histogram of positive recorded costs.
positive_costs = strikes.loc[
    strikes['Cost Total $'] > 0,
    'Cost Total $'
]

fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    positive_costs,
    bins=30,
    ax=ax
)

ax.set_xscale('log')

ax.set_title(
    'Positive Recorded Strike Costs Span a Very Wide Range'
)
ax.set_xlabel(
    'Positive recorded total cost in US dollars, logarithmic scale'
)
ax.set_ylabel('Number of reports')

plt.tight_layout()
plt.show()

CHART 4 (logarithmic version):
Positive recorded costs are strongly right-skewed and span a wide range, with
a small number of extremely expensive reports; the logarithmic axis makes the
smaller positive amounts visible.

Why use a logarithmic scale?
On a normal axis, a few extremely large costs stretch the scale and compress most observations near zero. A logarithmic scale spreads out smaller positive values while retaining the very large values.


___

**Requirement 9: Answer your own questions**

In [ ]:
# Values used in the Requirement 9 conclusions.
approach_reports = (
    strikes['Phase of flight']
    .eq('Approach')
    .sum()
)

daytime_reports = (
    strikes['Time of day']
    .eq('Day')
    .sum()
)

largest_phase_time = (
    pd.crosstab(
        strikes['Phase of flight'],
        strikes['Time of day']
    )
    .stack()
    .idxmax()
)

largest_phase_time_count = (
    pd.crosstab(
        strikes['Phase of flight'],
        strikes['Time of day']
    )
    .stack()
    .max()
)

zero_cost_reports = (
    strikes['Cost Total $']
    .eq(0)
    .sum()
)

zero_cost_percent = (
    strikes['Cost Total $']
    .eq(0)
    .mean()
    * 100
)

positive_costs = strikes.loc[
    strikes['Cost Total $'] > 0,
    'Cost Total $'
]

positive_cost_reports = len(positive_costs)
positive_cost_median = positive_costs.median()
positive_cost_mean = positive_costs.mean()
largest_recorded_cost = positive_costs.max()

yearly_reports = (
    strikes
    .groupby('Year')
    .size()
)

first_year = yearly_reports.index.min()
last_year = yearly_reports.index.max()
peak_year = yearly_reports.idxmax()

first_year_reports = yearly_reports.loc[first_year]
last_year_reports = yearly_reports.loc[last_year]
peak_year_reports = yearly_reports.loc[peak_year]

print('Approach reports:', approach_reports)
print('Daytime reports:', daytime_reports)
print(
    'Largest phase-time combination:',
    largest_phase_time,
    largest_phase_time_count
)

print('Zero-cost reports:', zero_cost_reports)
print(
    'Zero-cost percentage:',
    round(zero_cost_percent, 1)
)
print('Positive-cost reports:', positive_cost_reports)
print(
    'Median positive cost:',
    round(positive_cost_median, 1)
)
print(
    'Mean positive cost:',
    round(positive_cost_mean, 1)
)
print(
    'Largest recorded cost:',
    largest_recorded_cost
)

print(
    f'{first_year}:',
    first_year_reports
)
print(
    f'{last_year}:',
    last_year_reports
)
print(
    'Peak year:',
    peak_year,
    peak_year_reports
)

# Research questions and answers

## 1. During which flight phase and time of day were the most wildlife strikes reported?

The largest number of reports occurred during approach, with **4604**
reported strikes. Daytime had the largest overall number of reports, with
**5613**. The most common individual phase-and-time combination
was **('Approach', 'Night')**, with **2137** reports.

These figures show where reports are concentrated, but they do not prove that
approach or daytime flying is inherently the most dangerous. The dataset does
not contain the number of flights or the amount of time aircraft spent in each
phase and time category.

## 2. What do the recorded costs show?

A total of **9767 reports**, or **97.9%**, have a
recorded total cost of zero. Only **209 reports** contain a
positive recorded cost. Among those positive-cost reports, the median was
**$23325**, while the mean was **$193996.5**. The
largest individual recorded cost was **$7043545**.

The mean is higher than the median because a small number of exceptionally
expensive reports create a long right tail. The median is therefore more
representative of a typical positive recorded cost. However, a recorded zero
does not necessarily prove that an incident caused no financial loss because
the table does not say whether every zero was confirmed or whether some costs
were not reported.

## 3. How did the number of reported wildlife strikes change over time?

The dataset contains **462 reports in 1990** and
**627 reports in 2002**. The highest yearly count was
**1091 reports in 2001**. The line chart therefore shows
that the number of reports changed substantially during the period covered by
the dataset.

This does not prove that the underlying collision risk changed by the same
amount. The table contains reported incidents but does not include annual
flight totals, and changes in reporting practices or dataset coverage may also
affect the yearly counts.

___

**Requirement 10: Check the data and the source**

In [ ]:
print('Cleaned rows:', len(strikes))

print(
    'Missing damage descriptions:',
    strikes['Effect Amount of damage'].isna().sum()
)

print(
    'Missing speed values:',
    strikes['Speed IAS in knots'].isna().sum()
)

print(
    'Missing speed percentage:',
    (
        strikes['Speed IAS in knots'].isna().mean()
        * 100
    ).round(1)
)

print(
    'Zero recorded costs:',
    (strikes['Cost Total $'] == 0).sum()
)

print(
    'Zero recorded cost percentage:',
    (
        (strikes['Cost Total $'] == 0).mean()
        * 100
    ).round(1)
)

print(
    'Earliest date:',
    strikes['Flight Date'].min()
)

print(
    'Latest date:',
    strikes['Flight Date'].max()
)

# Data and source note

The original dataset contains 10,000 reports of aircraft-wildlife collisions
received by the US Federal Aviation Administration, while my cleaned table
contains 9,976 rows after 24 exact duplicate copies were removed. The data was
distributed through the Vega Datasets collection, while the stated original
source is the FAA Wildlife Strike Database. The reports cover the period from
8 January 1990 to 25 July 2002, and the dataset is identified as a US
government dataset in the public domain.

The table contains reported incidents rather than every aircraft-wildlife
collision that occurred. I do not know how the 10,000 records were selected
from the full FAA database, so I cannot assume that this extract is complete
or representative of all US wildlife strikes during the period.

The two main formally missing fields are `Effect Amount of damage` and
`Speed IAS in knots`. A missing damage description does not prove that no
damage occurred, and missing speeds were kept rather than replaced with
fictional values. The large number of zeros in `Cost Total $` is also
uncertain: zero may mean that no cost occurred, but the table does not state
whether every zero was confirmed or whether some costs were not reported.
The unexplained damage codes `B` and `C` were retained in the original column
and classified as `Unclear code` only in the derived analysis column.

The dataset does not contain flight totals, flight hours, aircraft altitude,
the number of animals struck, or a measure of reporting completeness.
Therefore, report counts cannot be interpreted as strike risk per flight.
I would not claim that approach is the most dangerous phase of flight simply
because it has the most reports, since the table does not show how many flights
entered each phase or how long aircraft spent there.


## Sources

- Vega Datasets, `birdstrikes.csv`: source, field summary, period and licence.
- FAA National Wildlife Strike Database: original reporting database.

___
___